In [9]:
import joblib
import numpy as np
from sentence_transformers import SentenceTransformer
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MultiLabelBinarizer

In [46]:
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
fast_model = joblib.load("models/fast_genre_head.pkl")
genres_name = np.load("dataset/genres_name.npy", allow_pickle=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [121]:
selected_column = [
    'budget','runtime','release_date','directors', 'cast',
    'genres','revenue','overview'
]

df = pd.read_csv('dataset/TMDB_IMDB_Movies_Dataset.csv', usecols=selected_column)
df.head(n=5)

,release_date,revenue,runtime,budget,overview,genres,directors,cast
0,2010-07-15,825532764,148,160000000,"Cobb, a skilled thief who commits corporate es...","Action, Science Fiction, Adventure",Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,2014-11-05,701729206,169,165000000,The adventures of a group of explorers who mak...,"Adventure, Drama, Science Fiction",Christopher Nolan,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,2008-07-16,1004558444,152,185000000,Batman raises the stakes in his war on crime. ...,"Drama, Action, Crime, Thriller",Christopher Nolan,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,2009-12-15,2923706026,162,237000000,"In the 22nd century, a paraplegic Marine is di...","Action, Adventure, Fantasy, Science Fiction",James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,2012-04-25,1518815515,143,220000000,When an unexpected enemy emerges and threatens...,"Science Fiction, Action, Adventure",Joss Whedon,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


In [122]:
# drop useless value
df = df.dropna(subset=['revenue', 'budget', 'release_date'])
df = df[(df['revenue'] > 10000) & (df['budget'] > 10000) & (df['overview'].notna()) & (df['overview'] != '')]

In [123]:
# convert date into month
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df = df.dropna(subset=['release_date'])

# sort by date to handle directors and casts revenue caluration
df = df.sort_values('release_date').reset_index(drop=True)

df['release_month'] = df['release_date'].dt.month
df = df.drop('release_date', axis=1)

df.head(n=5)

,revenue,runtime,budget,overview,genres,directors,cast,release_month
0,5773000,95,2221000,The daughter of King Neptune takes on human fo...,Fantasy,Herbert Brenon,"Annette Kellerman, William E. Shay, William We...",4
1,87028,50,16988,"Esra Kincaid takes land by force and, having t...","Romance, Western, Adventure",Cecil B. DeMille,"Bessie Barriscale, Jane Darwell, Dick La Reno,...",11
2,102224,45,15110,"A saloon hostess loves Ramerrez, a notorious h...","Western, Romance",Cecil B. DeMille,"Mabel Van Buren, Theodore Roberts, House Peter...",1
3,11000000,193,100000,"Two families, abolitionist Northerners the Sto...","Drama, History, War",D.W. Griffith,"Henry B. Walthall, Lillian Gish, Miriam Cooper...",2
4,137365,59,17311,"A venal, spoiled stockbroker's wife impulsivel...",Drama,Cecil B. DeMille,"Fannie Ward, Sessue Hayakawa, Jack Dean, James...",12


In [124]:
df['primary_director'] = df['directors'].fillna('Unknown').astype(str).str.split(',').str[0].str.strip()
df['lead_actor'] = df['cast'].fillna('Unknown').astype(str).str.split(',').str[0].str.strip()

# calculate last mean of last movie before this one
df['director_hist_rev'] = df.groupby('primary_director')['revenue'].transform(
    lambda x: x.expanding().mean().shift()
)
df['actor_hist_rev'] = df.groupby('lead_actor')['revenue'].transform(
    lambda x: x.expanding().mean().shift()
)

# make xgboost dont consider unknown as person
df.loc[df['primary_director'] == 'Unknown', 'director_hist_rev'] = np.nan
df.loc[df['lead_actor'] == 'Unknown', 'actor_hist_rev'] = np.nan

# new feature for xgboost 
df['is_debut_director'] = df['director_hist_rev'].isna().astype(int)
df['is_debut_actor'] = df['actor_hist_rev'].isna().astype(int)

global_median_rev = df['revenue'].median()

df['director_hist_rev'] = df['director_hist_rev'].fillna(global_median_rev)
df['actor_hist_rev'] = df['actor_hist_rev'].fillna(global_median_rev)

df = df.drop(columns=['directors', 'cast', 'primary_director', 'lead_actor'])

In [125]:
# encode our genres into top 12

genres_dummies = df['genres'].astype(str).str.get_dummies(sep=', ')
top_genres = genres_dummies.sum().sort_values(ascending=False).head(12).index
genres_dummies = genres_dummies[top_genres].add_prefix('genre_')

df = pd.concat([df, genres_dummies], axis=1)

In [126]:
# convert to use in NLP
nlp_genres = df['genres'].dropna().apply(lambda x: [i for i in x.split(', ')])
nlp_top_genres = nlp_genres.dropna().apply(lambda x: [genre for genre in x if genre in top_genres])

# encoding as formatting that BERT accept
mlb = MultiLabelBinarizer()
y_dataset = mlb.fit_transform(nlp_top_genres)
genres_name = mlb.classes_

print(y_dataset)
print(genres_name)

[[0 0 0 ... 0 0 0]
 [0 1 0 ... 1 0 0]
 [0 0 0 ... 1 0 0]
 ...
 [1 1 0 ... 0 0 1]
 [1 0 1 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
['Action' 'Adventure' 'Comedy' 'Crime' 'Drama' 'Family' 'Fantasy' 'Horror'
 'Mystery' 'Romance' 'Science Fiction' 'Thriller']


In [75]:
X_vectors = embedder.encode(df['overview'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/297 [00:00<?, ?it/s]

In [127]:
vibe_probs = np.array([p[:, 1] for p in fast_model.predict_proba(X_vectors)]).T
vibe_df = pd.DataFrame(vibe_probs, columns=[f"vibe_{g}" for g in genres_name])

In [128]:
df = pd.concat([df, vibe_df], axis=1)
df = df.drop(columns=['genres','overview'])
df.head(n=5)

,revenue,runtime,budget,release_month,director_hist_rev,actor_hist_rev,is_debut_director,is_debut_actor,genre_Drama,genre_Comedy,...,vibe_Comedy,vibe_Crime,vibe_Drama,vibe_Family,vibe_Fantasy,vibe_Horror,vibe_Mystery,vibe_Romance,vibe_Science Fiction,vibe_Thriller
0,5773000,95,2221000,4,15179302.0,15179302.0,1,1,0,0,...,0.077652,0.027446,0.004697,0.465334,0.100570,0.098104,0.029056,0.123095,0.117179,0.091361
1,87028,50,16988,11,15179302.0,15179302.0,1,1,0,0,...,0.074707,0.024376,0.090775,0.281133,0.021421,0.026131,0.006971,0.059085,0.045878,0.052594
2,102224,45,15110,1,87028.0,15179302.0,0,1,0,0,...,0.683906,0.178867,0.000297,0.383801,0.015265,0.006528,0.038898,0.476466,0.021861,0.123739
3,11000000,193,100000,2,15179302.0,15179302.0,1,1,1,0,...,0.022346,0.090913,0.107572,0.613073,0.015370,0.003826,0.006679,0.048237,0.319599,0.025023
4,137365,59,17311,12,94626.0,15179302.0,0,1,1,0,...,0.543389,0.106410,0.002249,0.399003,0.023526,0.007501,0.018310,0.083309,0.048791,0.145858


In [129]:
X = df.drop('revenue', axis=1)
# y = df['revenue']
y = np.log1p(df['revenue']) 

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.15, 
    shuffle=False
)

print(f"Training on {len(X_train)} historical movies...")
print(f"Testing on {len(X_test)} newer movies...")

Training on 8057 historical movies...
Testing on 1422 newer movies...


In [132]:
model = xgb.XGBRegressor(
    n_estimators=125,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    reg_lambda=10,
    # objective='reg:squarederror'
)

model.fit(X_train, y_train)

print("\n--- 6. Evaluating Performance ---")
log_predictions = model.predict(X_test)
real_dollar_predictions = np.expm1(log_predictions)
real_dollar_actuals = np.expm1(y_test)

mae = mean_absolute_error(real_dollar_actuals, real_dollar_predictions)
r2 = r2_score(real_dollar_actuals, real_dollar_predictions)


print(f"Test Set R-Squared: {r2:.2f} (1.0 is a perfect oracle)")
print(f"Mean Absolute Error: ${mae:,.2f} off per prediction")

print("\nTop 5 Most Important Features driving Box Office Revenue:")
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print(importance.head(5).to_string(index=False))

# SAVE THE MODEL FOR THE DASHBOARD
model.save_model("models/xgboost_box_office_model_nlp_version.json")


--- 6. Evaluating Performance ---
Test Set R-Squared: 0.47 (1.0 is a perfect oracle)
Mean Absolute Error: $67,682,447.14 off per prediction

Top 5 Most Important Features driving Box Office Revenue:
          Feature  Importance
           budget    0.502471
director_hist_rev    0.039855
      genre_Drama    0.038122
is_debut_director    0.037215
          runtime    0.030897
